# Anomaly Detection in Azure ML Studio

This proof of concept demonstrates how we can use Microsoft’s Azure AI platform, combined with custom-built analytics, to automatically identify unusual patterns or potential issues in data. It is designed to run within a secure, centralized environment, making it easy to deploy, monitor, and scale as needed 

### Requirements

1. You should already have created an Azure account, and created a [Subscription](https://techcommunity.microsoft.com/discussions/azure/understanding-azure-account-subscription-and-directory-/34800) and a [Workspace](https://learn.microsoft.com/en-us/azure/machine-learning/concept-workspace?view=azureml-api-2).
2. this exercise assumes that you've already downloaded the [Credit Card Fraud Detection](https://www.kaggle.com/datasets/mlg-ulb/creditcardfraud) dataset from kaggle, and stored this in 'Assets -> Data' inside your Azure account under the name, 'creditcard_fraud'.**

## Workflow

### Step 1: Import the neccessary libraries needed for our analysis


In [ ]:
# Step 1: Import Packages and Connect to your Azure Workspace
from azureml.core import Workspace, Dataset         # see https://pypi.org/project/azureml-core/
import pandas as pd                                 # see https://pandas.pydata.org/docs/
from sklearn.ensemble import IsolationForest        # see https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.IsolationForest.html
from sklearn.metrics import classification_report   # see https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html
from azureml.core.model import Model                # see https://docs.microsoft.com/en-us/python/api/azureml-core/azureml.core.model?view=azure-ml-py 

### Step 2: Data Ingestion — “Bringing the data into the system”

We start by loading historical transaction data into the system. This dataset contains thousands of past credit card transactions, including both normal and fraudulent ones.

This is the foundation—our model can only learn patterns based on the data we provide.

In [ ]:
# You only need to run this if you've imported this notebook to Azure AI Machine Learning Studio - Notebook,
# in which case you'll also need to upload the config.json file to the same directory as this notebook,
# and then execute this code to determine the current working directory.
import os
print("Current working directory:", os.getcwd())
print("Files in this directory:", os.listdir())
path='Users/[REPLACE-THIS-WITH-YOUR-USERNAME]/config.json'
ws = Workspace.from_config(path=path)
dataset = Dataset.get_by_name(ws, name='creditcard_fraud')
df = dataset.to_pandas_dataframe()
df.head()


### Step 3: Data Scaling — “Standardizing transaction values”

We normalize transaction amounts so that the system treats all features fairly.
Without this step, large transactions could disproportionately influence the model.

In [ ]:
df['Amount'] = (df['Amount'] - df['Amount'].mean()) / df['Amount'].std()
X = df.drop(columns=['Class', 'Time'])
y = df['Class']

### Step 4: Splitting the Data — “Training vs. validation”

We divide the dataset into:

Training data (to teach the model)
Test data (to validate performance)

This ensures the model performs well on unseen, real-world data—not just historical records.

In [ ]:
model = IsolationForest(contamination=0.0017, random_state=42)
model.fit(X)
y_pred = model.predict(X)
y_pred = [1 if x == -1 else 0 for x in y_pred]

### Step 5: Evaluating the Anomaly Detection Model

The table below is a summary of metrics that are calculated from the [confusion matrix](https://en.wikipedia.org/wiki/Confusion_matrix). It shows how well our model identified normal and fraudulent transactions:

| Metric       | What It Means                                                                 |
|--------------|--------------------------------------------------------------------------------|
| **Precision** | How often the model was *correct* when it said a transaction was fraud        |
| **Recall**    | How many of the *actual fraud cases* the model successfully found             |
| **F1-Score**  | A balance between precision and recall — like a combined performance score     |
| **Support**   | The number of examples in each group (normal or fraud) in the real data        |

#### Results Summary

| Class | Description           | Precision | Recall | F1-Score | Support |
|-------|------------------------|-----------|--------|----------|---------|
| `0`   | Normal transactions    | **1.00**  | **1.00** | **1.00**   | 284,315 |
| `1`   | Fraudulent transactions| **0.29**  | **0.28** | **0.28**   | 492     |

#### Interpretation (In Simple Terms)

- The model is **excellent at recognizing normal transactions** — it almost never makes a mistake with those.
- However, it **struggles to correctly catch fraud**:
  - When it says a transaction is fraud, it’s **only right 29% of the time**.
  - It **only finds 28% of the real fraud cases** — it misses most of them.

#### Overall Accuracy

- The model is **99.9% accurate**, but this is misleading.
- Because **fraud cases are very rare**, the model can look “perfect” just by saying everything is normal.
- That’s why we look at **precision**, **recall**, and **F1-score** for a fuller picture.


In [ ]:
# Step 5: Evaluate Model
print(classification_report(y, y_pred))

### Step 6 (Optional): Register the Model


In [ ]:
import joblib                                       # see https://joblib.readthedocs.io/en/latest/
                                                    #     Joblib is a set of tools to provide lightweight pipelining in Python
joblib.dump(model, 'isolation_forest.pkl')
Model.register(model_path='isolation_forest.pkl',
               model_name='creditcard_if_model',
               workspace=ws)


### Step 7: Visualize a Count of Predicted Anomalies

The chart below is a typical summarization of an anomaly detection analysis. It shows how many transactions the model predicted as **normal (0)** and **anomalies/fraud (1)**:

- **X-axis**: The prediction labels.
  - `0` means the model thinks the transaction is **normal**.
  - `1` means the model thinks the transaction is **fraud** or **anomalous**.
- **Y-axis**: The total number of transactions in each category.

#### How to Interpret This Chart

- You will (hopefully) see a **very tall bar for `0`** and a **very short bar for `1`**.
- This is because **fraud is rare** in the dataset (only 492 out of 284,807 transactions).
- The model is trained to detect outliers, so it **flags a small number of transactions as anomalies** (which is expected).
- If the number of predicted frauds is **close to the actual number** (around 500), that’s a good sign that the model is well-calibrated.

#### Why This Matters

- This simple chart gives a **quick health check** of how aggressive or conservative the model is in flagging anomalies.
- If the model predicts **too many anomalies**, it might be overreacting.
- If it predicts **almost none**, it might be too cautious — missing fraud cases.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Add predictions to the original dataframe
df['predicted_anomaly'] = y_pred

# Count of predicted anomalies
sns.countplot(x='predicted_anomaly', data=df)
plt.title('Count of Predicted Anomalies')
plt.xlabel('Anomaly (1) vs Normal (0)')
plt.ylabel('Count')
plt.show()


### Step 7 (continued): Visualize Transaction Amount by Prediction Class

The boxplot below compares the **amount of money** in transactions that the model predicted as **normal (0)** or **anomalous/fraud (1)**.

- **X-axis**: The model’s prediction.
  - `0` = predicted normal transaction
  - `1` = predicted fraud/anomaly
- **Y-axis**: The dollar **amount** of each transaction (standardized)

#### How to Interpret This Chart

- Each box shows how transaction amounts are distributed for each prediction class.
- The **line in the middle** of each box is the **median** transaction amount.
- The **height of the box** shows where most transaction amounts fall.
- **Dots outside the box** are **outliers** — unusual values far from the average.

#### What This Tells Us

- You may see that predicted frauds (`1`) tend to have **more extreme** or **variable amounts**.
- This could suggest that the model is flagging **unusually high or low transaction amounts** as suspicious.
- If the fraud predictions have a **much wider range**, it means the model may be reacting to extreme values — which is common in anomaly detection.

#### Usefulness

This chart helps you:
- Understand what kinds of amounts the model thinks are suspicious.
- Spot any bias in the model (e.g. only flagging large transactions).
- Decide whether you need to normalize, transform, or engineer features differently.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df, x='predicted_anomaly', y='Amount')
plt.title('Transaction Amount by Prediction Class')
plt.show()


### Step 7 (continued): SHAP Beeswarm Plot – Feature Importance for Anomaly Detection

The beeswarm plot below is generated using **SHAP** (SHapley Additive exPlanations). It helps explain **which features influenced the model's decisions**, and **how strongly**. We only analyze the first 100 transactions here in order to keep the visualization fast and readable.

#### How to Read the SHAP Beeswarm Plot

- **Each dot** represents a single transaction.
- **Each row** is one feature (like `V1`, `V2`, `Amount`, etc.).
- **Color** shows the feature value for that transaction:
  - **Red = high** value
  - **Blue = low** value
- **Horizontal position** shows **impact on the model’s prediction**:
  - Dots farther to the right **push the model toward predicting fraud**.
  - Dots farther to the left **push the model toward predicting normal**.

#### What This Tells Us

- The **topmost features** are the most important ones in the model’s decisions.
- For example, if `V14` is at the top and its red dots are far right, it means:
  - High values of `V14` increase the chance that the model flags a transaction as fraud.
- This plot helps us understand **why** the model flagged certain transactions as anomalies.

#### Why Use SHAP?

- SHAP adds transparency to the model, even for complex algorithms like Isolation Forest.
- Helps **build trust**, especially in sensitive tasks like fraud detection.
- Guides feature selection and **future model improvements**.

In [ ]:
import shap

explainer = shap.Explainer(model, X)
shap_values = explainer(X[:100])
shap.plots.beeswarm(shap_values)